In [3]:
import json
import pandas as pd

In [4]:
pd.set_option('display.max_colwidth', 1000)

In [115]:
gymLeadersFile = r"C:\Users\adam\Downloads\Reformed 1.0 Gym Leaders.Json"
pokemonFile = r"C:\Users\adam\Desktop\Projects\Pokemon\Reformed Platinum\Lumi 2.1F Pokemon Dump.json"

In [132]:
df_gymLeaders = pd.read_json(gymLeadersFile).T
df_pokemon = pd.read_json(pokemonFile)

In [133]:
name_data = df_pokemon['readOnly'].apply(pd.Series)['name']
form_Id = df_pokemon['readOnly'].apply(pd.Series)['formID']
typing_data = df_pokemon['typing']

In [134]:
df_pokemon = pd.DataFrame({
    'name': name_data,
    'form': form_Id,
    'typing': typing_data
})
typing_expanded  = df_pokemon['typing'].apply(lambda x: pd.Series(x + [None]*(2-len(x))))
df_pokemon = pd.concat([df_pokemon, typing_expanded.rename(columns={0: 'type1', 1: 'type2'})], axis=1)
df_pokemon.drop('typing', axis=1, inplace=True)


In [135]:
typeReplacements = {
                "Bug": "Fairy",
                "Dark": "Fire",
                "Dragon": "Psychic",
                "Electric": "Fighting",
                "Fairy": "Flying",
                "Fighting": "Rock",
                "Fire": "Electric",
                "Flying": "Ghost",
                "Ghost": "Poison",
                "Grass": "Dark",
                "Ground": "Grass",
                "Ice": "Ground",
                "Normal": "Bug",
                "Water": "Normal",
                "Poison": "Ice",
                "Psychic": "Steel",
                "Steel": "Water",
                "Rock": "Dragon"
}

In [136]:
df_pokemon['type1'].replace(typeReplacements, inplace=True)
df_pokemon['type2'].replace(typeReplacements, inplace=True)

C:\Users\adam\AppData\Local\Temp\ipykernel_16828\2187847538.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_pokemon['type1'].replace(typeReplacements, inplace=True)
C:\Users\adam\AppData\Local\Temp\ipykernel_16828\2187847538.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

In [137]:
df_pokemon

,name,form,type1,type2
0,Egg,0,Bug,None
1,Bulbasaur,0,Dark,Ice
2,Ivysaur,0,Dark,Ice
3,Venusaur,0,Dark,Ice
4,Charmander,0,Electric,None
...,...,...,...,...
1461,Palafin,1,Normal,Water
1462,Tatsugiri,1,Normal,Water
1463,Tatsugiri,2,Normal,Water
1464,Dudunsparce,1,Normal,Water


In [138]:
df_gymLeaders.drop(['readOnly','trainerTypeID','colorID','fightType','arenaID','effectID','gold','useItems','hpRecoverFlag','giftItem','nameLabel','aiFlags'],axis=1,inplace=True)

In [139]:
df_gymLeaders = df_gymLeaders.explode("party")
df_gymLeaders = df_gymLeaders.join(df_gymLeaders['party'].apply(pd.Series))
df_gymLeaders.drop(['party', 'shiny', 'ballID', 'seal'], axis=1, inplace=True)
df_gymLeaders = df_gymLeaders.reset_index()
df_gymLeaders = df_gymLeaders.astype(str).drop_duplicates()

In [140]:
df_pokemon['form']

0       0
1       0
2       0
3       0
4       0
       ..
1461    1
1462    1
1463    2
1464    1
1465    1
Name: form, Length: 1466, dtype: object

In [141]:
df_gymLeaders['formID']

0       0
1       0
2       0
3       0
4       0
       ..
1837    0
1838    0
1839    0
1840    0
1841    0
Name: formID, Length: 312, dtype: object

In [152]:
df_gymLeaders['species'] = df_gymLeaders['species'].str.strip()
df_gymLeaders['formID'] = df_gymLeaders['formID'].astype(str).str.strip()  # Convert to string if needed

# Remove whitespaces from the 'name' and 'form' columns in df_pokemon
df_pokemon['name'] = df_pokemon['name'].str.strip()
df_pokemon['form'] = df_pokemon['form'].astype(str).str.strip()  # Convert to string if needed

In [153]:
df_gymLeadersInfo = pd.merge(
    df_gymLeaders,
    df_pokemon,
    left_on=['species', 'formID'],
    right_on=['name', 'form'],
    how='inner'
)

In [155]:
df_gymLeadersInfo.to_csv(r"C:\Users\adam\Downloads\Reformed 1.0 Gym Leaders.csv", index=True)